# Detección de lavado de dinero (AML) con el dataset real de IBM, vía Hugging Face

**Versión independiente de `Fraud_Detection.ipynb`** — mismo tipo de pregunta (clustering, clasificación, detección de anomalías por reconstrucción), pero sobre un dataset real de investigación en vez de `bank_transactions` sintético, y sin depender de BigQuery ni de credenciales de Kaggle.

## El dataset

[**IBM Transactions for Anti Money Laundering (AML)**](https://www.kaggle.com/datasets/ealtman2019/ibm-transactions-for-anti-money-laundering-aml) — generado por IBM Research con un simulador multiagente (`AMLSim`), 100% sintético (ninguna persona real detrás). Licencia del dato original: **CDLA-Sharing-1.0** (exige atribución a IBM si se republica el dato; no bloquea su uso en clase). Usamos la variante **HI-Small**: ~5.08M transacciones, 10 días, **~0.10% marcadas como lavado** — un desbalance mucho más extremo y realista que el de `bank_transactions.csv`.

## Por qué Hugging Face y no Kaggle

El archivo `HI-Small_Trans.csv` (476 MB) está espejado, público y sin login, en
[`bbfizp/AMLSim-HI-Small`](https://huggingface.co/datasets/bbfizp/AMLSim-HI-Small).
Con `huggingface_hub` se descarga con una sola línea, sin pedir token — cero fricción
de credenciales en clase, mismo criterio que ya usamos al preferir Frankfurter sobre
Banxico en `recursos/etl-tipo-cambio/`. **Nota de atribución:** el espejo en HF declara
licencia Apache-2.0 para su re-empaquetado, pero el dato en sí sigue siendo de IBM bajo
CDLA-Sharing-1.0 — si compartes este dataset fuera del curso, atribuye a IBM.

⚠️ Este espejo **no incluye** `HI-Small_Patterns.txt` (la lista de qué transacciones
pertenecen a cada patrón de lavado: Fan-In, Fan-Out, Gather-Scatter, Cycle, Random). Si
quieres esa verdad de referencia para validar los clusters del Bloque A, hay que bajarla
de Kaggle directo (requiere cuenta) — aquí no la usamos.

## El cluster: **sección B.1** de MLOPS/README.md, no la sección B

Este notebook necesita `huggingface_hub` (y por transitividad `datasets`/`transformers`/`torch`),
que **no** están en el cluster de la sección B (el que usan `Fraud_Detection`, `PySpark_Intro`,
etc.). Hace falta el cluster con `hugging_face_deps.sh` como initialization action:

```bash
export PROJECT_ID=$(gcloud config get-value project)
export REGION=us-central1
export ZONE=us-central1-a
export CLUSTER_NAME=hf-learning-cluster
export BUCKET_NAME=<TU-BUCKET>

gcloud dataproc clusters create $CLUSTER_NAME \
    --project=$PROJECT_ID --region=$REGION --zone=$ZONE \
    --image-version=2.1-ubuntu20 \
    --master-machine-type=n1-standard-2 --master-boot-disk-size=50 \
    --num-workers=2 --worker-machine-type=n1-standard-4 --worker-boot-disk-size=100 \
    --initialization-actions=gs://$BUCKET_NAME/hugging_face_deps.sh \
    --metadata=PIP_PACKAGES="transformers datasets torch" \
    --enable-component-gateway \
    --max-idle=1h --max-age=3h \
    --scopes='https://www.googleapis.com/auth/cloud-platform' \
    --properties="dataproc:jupyter.notebook.gcs.dir=gs://$BUCKET_NAME/notebooks"
```

**Compatibilidad verificada (sin correrlo en el cluster real):** `hugging_face_deps.sh`
instala `transformers datasets torch pandas numpy` con `pip install` global en los 3 nodos
(no usa `--single-node`, así que corre en master y en ambos workers). `huggingface_hub` se
instala automáticamente como dependencia de `datasets` — no hace falta tocar el script.
La descarga en sí ocurre en el **driver** (el notebook corre en el master), así que basta
con que el master tenga salida a internet — igual que ya necesita para `pip install` en el
init action.

Tres bloques, igual que en `Fraud_Detection.ipynb`, adaptados al esquema del dataset real:

| Bloque | Técnica | Tipo de aprendizaje | Pregunta |
|---|---|---|---|
| **A** | K-Means + PCA | No supervisado | ¿Hay grupos naturales de transacciones? |
| **B** | Random Forest | Supervisado | ¿Podemos predecir el lavado con estas columnas? |
| **C** | Reconstrucción con PCA | No supervisado (aprende solo de lo normal) | ¿Qué tan raro es cada movimiento? |

In [ ]:
# Parche de compatibilidad para el Jupyter de Dataproc: matplotlib llama a RcParams._get,
# que no existe en este entorno. Debe ejecutarse ANTES de importar pyplot.
import matplotlib
_RcParams = type(matplotlib.rcParams)
if not hasattr(_RcParams, '_get'):
    _RcParams._get = _RcParams.get

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from pyspark.sql import SparkSession, functions as F
from pyspark.sql.functions import col, hour, dayofweek
from pyspark.ml import Pipeline
from pyspark.ml.feature import VectorAssembler, StandardScaler, StringIndexer, OneHotEncoder, PCA
from pyspark.ml.functions import vector_to_array
from pyspark.ml.clustering import KMeans
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator, ClusteringEvaluator

In [ ]:
spark = SparkSession.builder.appName("AML_HuggingFace").getOrCreate()

## 1. Traer el dato: Hugging Face → disco local del driver → Cloud Storage

Tres pasos, cada uno con su propia bandera para poder saltarlo si ya corriste esta celda antes
(no vuelve a descargar ni a subir 476 MB cada vez que reinicias el kernel):

1. `hf_hub_download` trae el CSV al caché local del driver (por defecto `~/.cache/huggingface`).
2. Lo subimos a tu bucket con `gsutil` — así el dato vive en Storage, igual que el resto del curso,
   y no depende de que el driver siga vivo.
3. Spark lo lee **directo de `gs://`** con el conector de Cloud Storage que ya trae Dataproc —
   sin pasar por BigQuery.

In [ ]:
BUCKET = "big-data-lunes-20260223"                       # ajusta a tu bucket
RUTA_GCS = f"gs://{BUCKET}/raw/aml/HI-Small_Trans.csv"

DESCARGAR_DE_HF = True   # False si ya subiste el archivo a RUTA_GCS en una corrida anterior

In [ ]:
if DESCARGAR_DE_HF:
    from huggingface_hub import hf_hub_download

    print("Descargando HI-Small_Trans.csv desde Hugging Face (~476 MB, tarda unos minutos)...")
    ruta_local = hf_hub_download(
        repo_id="bbfizp/AMLSim-HI-Small",
        repo_type="dataset",
        filename="HI-Small_Trans.csv",
    )
    print(f"Descargado en el driver: {ruta_local}")

In [ ]:
if DESCARGAR_DE_HF:
    import subprocess
    print(f"Subiendo a {RUTA_GCS} ...")
    subprocess.run(["gsutil", "cp", ruta_local, RUTA_GCS], check=True)
    print("Listo.")

## 2. Leer con un esquema explícito (no confiar en el encabezado)

El CSV real trae la columna `Account` **dos veces** (una para la cuenta origen, otra para
la destino) — un encabezado ambiguo. Pandas la renombra sola a `Account.1`, pero Spark no
lo hace de forma consistente entre versiones (lo probamos en local: en Spark 4.x renombra
con una advertencia; en 3.x puede fallar directo). La forma robusta es ignorar el
encabezado y asignar el esquema nosotros mismos, en el orden documentado del dataset.

In [ ]:
COLUMNAS = ["timestamp", "from_bank", "from_account", "to_bank", "to_account",
            "amount_received", "receiving_currency", "amount_paid", "payment_currency",
            "payment_format", "is_laundering"]

df_raw = (spark.read.option("header", False).csv(RUTA_GCS)
          .toDF(*COLUMNAS)
          .filter(col("timestamp") != "Timestamp"))   # la fila de encabezado quedó como dato

df_raw.printSchema()
df_raw.show(5, truncate=False)

In [ ]:
n_total = df_raw.count()
n_laundering = df_raw.filter(col("is_laundering") == "1").count()
print(f"Transacciones: {n_total:,} | lavado: {n_laundering:,} ({n_laundering / n_total:.3%})")
print("Un modelo que siempre diga 'normal' ya acierta", f"{1 - n_laundering / n_total:.3%}")

## 3. Features

Dos señales que `bank_transactions.csv` no tiene, por venir de un par de cuentas con dos
montos y dos monedas:

- **`mismo_banco`** — si el banco origen y destino son el mismo. Mover dinero entre bancos
  distintos es más común en cadenas de lavado (dificulta el rastreo).
- **`moneda_distinta`** — si `Receiving Currency` ≠ `Payment Currency`. Un cambio de divisa
  en la misma transacción es una técnica clásica de *layering*.

In [ ]:
df_base = (df_raw
    .withColumn("amount", col("amount_paid").cast("double"))
    .withColumn("ts", F.to_timestamp("timestamp", "yyyy/MM/dd HH:mm"))
    .withColumn("hour", hour("ts"))
    .withColumn("day", dayofweek("ts"))
    .withColumn("mismo_banco", (col("from_bank") == col("to_bank")).cast("string"))
    .withColumn("moneda_distinta", (col("receiving_currency") != col("payment_currency")).cast("string"))
    .withColumn("label_real", col("is_laundering").cast("double"))
    .select("amount", "hour", "day", "from_bank", "to_bank", "payment_format",
            "mismo_banco", "moneda_distinta", "label_real")
    .na.drop())

df_base.show(5)

# BLOQUE A — Clustering (segmentación de riesgo)

Mismo patrón que en `Fraud_Detection.ipynb`: sin usar la etiqueta, agrupar transacciones
parecidas y ver si el lavado se concentra en algún grupo. `from_bank`/`to_bank` tienen
**miles** de valores distintos (son IDs de cuenta bancaria simulada, no categorías chicas
como país) — `handleInvalid="skip"` en el indexador evita que una categoría nueva en test
tumbe el pipeline, pero con tantas categorías el One-Hot infla mucho el vector. Por eso
aquí solo indexamos `payment_format` (pocas categorías reales) y dejamos `from_bank`/
`to_bank` fuera de A y B — sí importan, pero como grafo (cuenta-a-cuenta), no como
categoría plana; es quien las ataca bien un modelo de grafos, no K-Means ni Random Forest.

In [ ]:
cat_cols = ["payment_format", "mismo_banco", "moneda_distinta"]
indexers = [StringIndexer(inputCol=c, outputCol=c + "_idx", handleInvalid="skip") for c in cat_cols]
encoder = OneHotEncoder(inputCols=[c + "_idx" for c in cat_cols], outputCols=[c + "_vec" for c in cat_cols])
assembler = VectorAssembler(
    inputCols=["amount", "hour", "day"] + [c + "_vec" for c in cat_cols],
    outputCol="features_raw"
)
scaler = StandardScaler(inputCol="features_raw", outputCol="features_scaled")

In [ ]:
%%time
prep_pipeline = Pipeline(stages=indexers + [encoder, assembler, scaler])
prep_model = prep_pipeline.fit(df_base)
df_final = prep_model.transform(df_base)
df_final.select("amount", "hour", "day", "label_real", "features_scaled").show(5, truncate=60)

### ¿Cuántos grupos? El método del codo

In [ ]:
%%time
K_MAX = 6
cost = []
print(f"Calculando el método del codo para K de 2 a {K_MAX - 1}...")
for k in range(2, K_MAX):
    modelo_k = KMeans(featuresCol="features_scaled", k=k, seed=42).fit(df_final)
    cost.append(modelo_k.summary.trainingCost)
    print(f"  K={k}  costo={cost[-1]:,.0f}")

In [ ]:
df_elbow = pd.DataFrame({'K': list(range(2, K_MAX)), 'WSS': cost})
fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(df_elbow['K'], df_elbow['WSS'], marker='o', linestyle='-', color='#1f77b4', linewidth=2, markersize=6)
ax.set_title('Método del Codo (Selección de K)', fontsize=14, pad=15, fontweight='bold')
ax.set_xlabel('K'); ax.set_ylabel('WSS (costo de entrenamiento)')
ax.grid(True, linestyle='--', alpha=0.5, color='#ccc')
ax.set_xticks(df_elbow['K'])
ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
plt.show()

In [ ]:
%%time
k_optimo = 4
kmeans_final = KMeans(featuresCol="features_scaled", k=k_optimo, predictionCol="cluster", seed=42)
model_final = kmeans_final.fit(df_final)
df_res = model_final.transform(df_final)

silueta = ClusteringEvaluator(featuresCol="features_scaled", predictionCol="cluster",
                              metricName="silhouette").evaluate(df_res.sample(0.05, seed=42))
print(f"Silueta (muestra 5%) con K={k_optimo}: {silueta:.3f}")

print("\nLavado por cluster (¿se concentra en alguno?):")
(df_res.groupBy("cluster")
       .agg(F.count("*").alias("transacciones"),
            F.round(F.avg("label_real") * 100, 4).alias("pct_lavado"))
       .orderBy("cluster").show())

# BLOQUE B — Clasificación (Random Forest)

Con un desbalance de ~0.10% (10 a 100 veces más extremo que `bank_transactions.csv`), el
peso inverso a la frecuencia es todavía más importante: sin él, el modelo puede converger
a "siempre normal" y verse bien en accuracy mientras el recall de lavado es cero.

In [ ]:
%%time
n = df_final.count()
n_pos = df_final.filter(col("label_real") == 1).count()
n_neg = n - n_pos
print(f"Lavado: {n_pos:,} de {n:,} ({n_pos / n:.4%}). Un modelo que siempre diga 'normal' acierta {n_neg / n:.4%}.")

df_clf = df_final.withColumn("peso", F.when(col("label_real") == 1, n / (2.0 * n_pos)).otherwise(n / (2.0 * n_neg)))
train, test = df_clf.randomSplit([0.8, 0.2], seed=42)

rf = RandomForestClassifier(labelCol="label_real", featuresCol="features_raw", weightCol="peso",
                            numTrees=50, seed=42)
model_rf = rf.fit(train)
preds = model_rf.transform(test)

auc_roc = BinaryClassificationEvaluator(labelCol="label_real", metricName="areaUnderROC").evaluate(preds)
auc_pr  = BinaryClassificationEvaluator(labelCol="label_real", metricName="areaUnderPR").evaluate(preds)
print(f"\n>>> AUC-ROC = {auc_roc:.4f} | AUC-PR = {auc_pr:.4f} (azar: AUC-ROC 0.5, AUC-PR ≈ {n_pos / n:.4f}) <<<")

**Cómo leerlo:** con un AUC-PR tan bajo como línea base (~0.001), incluso un modelo
modesto que la supere por un buen margen ya es útil — en un problema tan desbalanceado,
comparar contra la línea base importa más que el número absoluto.

# BLOQUE C — Detección de anomalías por error de reconstrucción

Mismo autoencoder lineal (PCA, ajustado solo con transacciones normales de train) que en
`Fraud_Detection.ipynb`. Reglas: split antes de ajustar, el "encoder" solo ve normales de
train, y el umbral se define con datos de train, nunca con el conjunto que se evalúa.

In [ ]:
%%time
train_c, test_c = df_base.randomSplit([0.8, 0.2], seed=42)

scaler_c = StandardScaler(inputCol="features_raw", outputCol="features_scaled", withMean=True, withStd=True)
prep_c = Pipeline(stages=indexers + [encoder, assembler, scaler_c]).fit(train_c)
train_f = prep_c.transform(train_c)
test_f = prep_c.transform(test_c)

K_BOTTLENECK = 6
normales_train = train_f.filter(col("label_real") == 0)
ae = PCA(k=K_BOTTLENECK, inputCol="features_scaled", outputCol="codigo").fit(normales_train)
print(f"Variables de entrada: {len(train_f.select('features_scaled').first()[0])} -> cuello de botella: {K_BOTTLENECK}")
print(f"Información que conserva el cuello de botella: {sum(ae.explainedVariance):.1%}")

def norma2(c):
    return F.aggregate(vector_to_array(c), F.lit(0.0), lambda acc, x: acc + x * x)

def puntuar(d):
    d = ae.transform(d)
    return d.withColumn("anomaly_score", F.sqrt(F.greatest(norma2("features_scaled") - norma2("codigo"), F.lit(0.0))))

train_scored = puntuar(train_f)
df_predictions = puntuar(test_f).persist()

In [ ]:
umbral = train_scored.filter(col("label_real") == 0).approxQuantile("anomaly_score", [0.95], 0.001)[0]
print(f"Umbral (percentil 95 de normales en train): {umbral:.4f}")

df_predictions = df_predictions.withColumn("alerta", (col("anomaly_score") > umbral).cast("int"))
print("\nMatriz de confusión (filas = realidad, columnas = alerta):")
df_predictions.groupBy("label_real").pivot("alerta", [0, 1]).count().na.fill(0).orderBy("label_real").show()

cm = {(r["label_real"], r["alerta"]): r["count"] for r in df_predictions.groupBy("label_real", "alerta").count().collect()}
tp, fn, fp = cm.get((1.0, 1), 0), cm.get((1.0, 0), 0), cm.get((0.0, 1), 0)
print(f"Lavado detectado (recall): {tp / max(tp + fn, 1):.1%} | de las alertas, lavado real (precisión): {tp / max(tp + fp, 1):.1%}")

roc_ae = BinaryClassificationEvaluator(labelCol="label_real", rawPredictionCol="anomaly_score", metricName="areaUnderROC").evaluate(df_predictions)
pr_ae = BinaryClassificationEvaluator(labelCol="label_real", rawPredictionCol="anomaly_score", metricName="areaUnderPR").evaluate(df_predictions)
print(f"AUC-ROC = {roc_ae:.4f} | AUC-PR = {pr_ae:.4f}")

## Para practicar

1. **Grafo:** `from_bank`/`from_account`/`to_bank`/`to_account` forman una red de
   transferencias — construye el grafo (nodos = cuentas, aristas = transacciones) y
   calcula el grado de cada cuenta. ¿Las cuentas con grado muy alto tienen más lavado?
2. **Comparación directa:** corre `Fraud_Detection.ipynb` y este notebook con las mismas
   métricas (AUC-PR del Bloque B y C) y compara qué tan distinto es detectar fraude en
   datos sintéticos de curso vs. en un benchmark de investigación real.
3. **`HI-Small` vs `LI-Small`:** cambia `filename="HI-Small_Trans.csv"` por
   `"LI-Small_Trans.csv"` (mismo repo si está espejado, o descárgalo de Kaggle) y compara
   cómo cambian los resultados con una proporción de lavado distinta.
4. **Patrones reales:** si te animas a usar Kaggle directo, trae `HI-Small_Patterns.txt`
   y compara los clusters del Bloque A contra los patrones reales (Fan-In, Cycle, etc.)
   en vez de solo mirar el % de lavado por cluster.

In [ ]:
spark.stop()